# EcoData · Unidad 1 · Dimensión Aire (Lanzeloth Vargas Marichi)
**Pregunta de la dimensión:** ¿cuál es el nivel histórico de PM10/PM2.5 (µg/m³) en Juliaca, cuál su media mensual y cuánto cambiará en la próxima hora?
**Instrumento:** Open-Meteo Air Quality (Juliaca, lat −15.5, lon −70.1, 3 826 m). **Histórico base:** año 2023 (8 760 h).
Pipeline: Bronze (JSON crudo) → Silver (limpio) → Gold (Parquet particionado por mes) → ML (MLlib).

In [1]:
import os, json, time, getpass, platform
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
spark=(SparkSession.builder.master("local[2]").appName("EcoData-U1-Aire")
       .config("spark.ui.enabled","false").config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Evidencia | usuario:",getpass.getuser(),"| equipo:",platform.node(),"| reloj:",time.strftime("%Y-%m-%d %H:%M:%S"))
print("Spark",spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 20:13:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Evidencia | usuario: root | equipo: d14c8d16aa21 | reloj: 2026-11-19 20:13:31
Spark 4.2.0


## 1 · Arquitectura (S1)
Lambda: la **capa batch** (este notebook) entrena el modelo sobre el histórico; la **capa de velocidad** (U2: Kafka → Spark Streaming) solo consume el modelo guardado. Entre ambas capas cruza únicamente el modelo `modelo_aire_u1`.

## 2 · Extracción y transformaciones PySpark (S2)

In [2]:
ORIGEN="aq_juliaca_2023.json"      # copia local de la respuesta de Open-Meteo (reproducible sin red)
# Alternativa en vivo (requiere red):
# https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-15.5&longitude=-70.1&hourly=pm10,pm2_5&start_date=2023-01-01&end_date=2023-12-31
h=json.load(open(ORIGEN))["hourly"]
os.makedirs("bronze",exist_ok=True)
pd.DataFrame({"time":h["time"],"pm10":h["pm10"],"pm2_5":h["pm2_5"]}).to_csv("bronze/aire_juliaca_2023.csv",index=False)
esquema=StructType([StructField("time",StringType(),True),StructField("pm10",DoubleType(),True),StructField("pm2_5",DoubleType(),True)])
bronze=spark.read.csv("bronze/aire_juliaca_2023.csv",header=True,schema=esquema)
print("Bronze filas:",bronze.count()); bronze.printSchema(); bronze.show(5)

Bronze filas: 8760
root
 |-- time: string (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)

+----------------+----+-----+
|            time|pm10|pm2_5|
+----------------+----+-----+
|2023-01-01T00:00| 0.0|  0.0|
|2023-01-01T01:00| 0.3|  0.2|
|2023-01-01T02:00| 0.5|  0.4|
|2023-01-01T03:00| 0.4|  0.3|
|2023-01-01T04:00| 0.4|  0.3|
+----------------+----+-----+
only showing top 5 rows


In [3]:
t=(bronze.withColumn("ts",F.to_timestamp("time"))
         .withColumn("mes",F.date_format("ts","yyyy-MM"))
         .withColumn("hora",F.hour("ts"))
         .withColumn("nivel_riesgo",F.when(F.col("pm2_5")>=50,"Alto").when(F.col("pm2_5")>=25,"Moderado").otherwise("Bajo")))
# Evaluación perezosa: hasta aquí NO se ha ejecutado nada
t0=time.time(); plan=t.filter(F.col("pm2_5")>10); print("transformación definida en %.4f s (sin acción)"%(time.time()-t0))
t0=time.time(); n=plan.count(); print("acción count() = %d en %.2f s"%(n,time.time()-t0))

transformación definida en 0.0234 s (sin acción)
acción count() = 626 en 0.28 s


In [4]:
mensual=(t.groupBy("mes").agg(F.count("*").alias("horas"),F.round(F.avg("pm2_5"),2).alias("pm25_media"),
          F.round(F.expr("percentile_approx(pm2_5,0.95)"),2).alias("pm25_p95"),F.round(F.max("pm2_5"),1).alias("pm25_max"),
          F.round(F.avg("pm10"),2).alias("pm10_media")).orderBy("mes"))
mensual.show(12,truncate=False)
t.groupBy("nivel_riesgo").agg(F.count("*").alias("horas"),F.round(F.avg("pm2_5"),2).alias("pm25_media")).show()

+-------+-----+----------+--------+--------+----------+
|mes    |horas|pm25_media|pm25_p95|pm25_max|pm10_media|
+-------+-----+----------+--------+--------+----------+
|2023-01|744  |1.63      |3.5     |5.8     |2.4       |
|2023-02|672  |4.77      |10.3    |13.5    |6.84      |
|2023-03|744  |4.33      |8.6     |11.6    |6.26      |
|2023-04|720  |5.73      |9.5     |16.4    |8.31      |
|2023-05|744  |5.87      |11.3    |21.6    |8.57      |
|2023-06|720  |5.21      |9.3     |12.3    |7.61      |
|2023-07|744  |5.94      |12.3    |17.8    |8.7       |
|2023-08|744  |6.82      |12.9    |32.7    |10.03     |
|2023-09|720  |5.79      |11.3    |35.2    |8.46      |
|2023-10|744  |5.61      |11.6    |24.6    |8.16      |
|2023-11|720  |5.48      |11.4    |27.6    |8.08      |
|2023-12|744  |4.95      |11.7    |17.0    |8.13      |
+-------+-----+----------+--------+--------+----------+

+------------+-----+----------+
|nivel_riesgo|horas|pm25_media|
+------------+-----+----------+
|      

In [5]:
print("=== explain(True) de la agregación mensual ===")
mensual.explain(True)

=== explain(True) de la agregación mensual ===
== Parsed Logical Plan ==
'Sort ['mes ASC NULLS FIRST], true
+- Aggregate [mes#25], [mes#25, count(1) AS horas#40L, round(avg(pm2_5#2), 2) AS pm25_media#41, round(percentile_approx(pm2_5#2, cast(0.95 as double), 10000, 0, 0), 2) AS pm25_p95#42, round(max(pm2_5#2), 1) AS pm25_max#43, round(avg(pm10#1), 2) AS pm10_media#44]
   +- Project [time#0, pm10#1, pm2_5#2, ts#24, mes#25, hora#26, CASE WHEN (pm2_5#2 >= cast(50 as double)) THEN Alto WHEN (pm2_5#2 >= cast(25 as double)) THEN Moderado ELSE Bajo END AS nivel_riesgo#27]
      +- Project [time#0, pm10#1, pm2_5#2, ts#24, mes#25, hour(ts#24, Some(UTC)) AS hora#26]
         +- Project [time#0, pm10#1, pm2_5#2, ts#24, date_format(ts#24, yyyy-MM, Some(UTC)) AS mes#25]
            +- Project [time#0, pm10#1, pm2_5#2, to_timestamp(time#0, None, TimestampType, Some(UTC), true) AS ts#24]
               +- Relation [time#0,pm10#1,pm2_5#2] csv

== Analyzed Logical Plan ==
mes: string, horas: bigint, pm

In [6]:
# RDD: horas por nivel de riesgo con map/reduceByKey
rdd=t.select("nivel_riesgo").rdd.map(lambda r:(r[0],1)).reduceByKey(lambda a,b:a+b)
print(sorted(rdd.collect()))

[Stage 13:>                                                         (0 + 1) / 1]

[('Bajo', 8747), ('Moderado', 13)]


## 3 · Calidad de datos y particionamiento (S3)

In [7]:
req={"time","pm10","pm2_5"}; assert req<=set(bronze.columns),"faltan columnas"
n0=bronze.count()
nulos=bronze.select([F.sum(((F.col(c).isNull())|(F.col(c).cast("string")==""))  .cast("int")).alias(c) for c in bronze.columns]).first().asDict()
dups=n0-bronze.dropDuplicates(["time"]).count()
rango=bronze.filter((F.col("pm2_5")<0)|(F.col("pm10")<0)).count(); redond=bronze.filter(F.col("pm2_5")>F.col("pm10")).count(); print("pm2_5>pm10 por redondeo (se conservan):",redond)
print("ANTES  filas=%d | nulos=%s | duplicados(time)=%d | valores negativos=%d"%(n0,nulos,dups,rango))
w=Window.partitionBy("time").orderBy(F.col("pm2_5").desc_nulls_last())      # deduplicación determinista
silver=(bronze.dropna(subset=["pm10","pm2_5"]).withColumn("rn",F.row_number().over(w)).filter("rn=1").drop("rn")
        .filter((F.col("pm2_5")>=0)&(F.col("pm10")>=0))   # solo valores negativos son inválidos; pm10<pm2_5 por redondeo se conserva (se reporta)
        .withColumn("ts",F.to_timestamp("time")).withColumn("mes",F.date_format("ts","yyyy-MM")))
n1=silver.count()
print("DESPUÉS filas=%d | descartadas=%d"%(n1,n0-n1))
# huecos horarios
tt=silver.orderBy("ts").select(F.col("ts").cast("long").alias("s")).toPandas().s.values
print("huecos horarios (>1 h entre registros):",int((np.diff(tt)>3600).sum()))

pm2_5>pm10 por redondeo (se conservan): 12
ANTES  filas=8760 | nulos={'time': 0, 'pm10': 0, 'pm2_5': 0} | duplicados(time)=0 | valores negativos=0
DESPUÉS filas=8760 | descartadas=0


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


huecos horarios (>1 h entre registros): 0


In [8]:
RUTA="gold/aire_mensual"
(silver.select("ts","pm10","pm2_5","mes").write.mode("overwrite").partitionBy("mes").parquet(RUTA))
back=spark.read.parquet(RUTA)
assert back.count()==n1, "la lectura de vuelta no coincide"
print("round-trip OK:",back.count(),"filas |",len([d for d in os.listdir(RUTA) if d.startswith("mes=")]),"particiones")
print(sorted(d for d in os.listdir(RUTA) if d.startswith("mes="))[:4],"...")
back.filter(F.col("mes")=="2023-06").explain()   # buscar PartitionFilters

round-trip OK: 8760 filas | 12 particiones
['mes=2023-01', 'mes=2023-02', 'mes=2023-03', 'mes=2023-04'] ...
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [ts#227,pm10#228,pm2_5#229,mes#230] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/u1-aire/gold/aire_mensual], PartitionFilters: [isnotnull(mes#230), (mes#230 = 2023-06)], PushedFilters: [], ReadSchema: struct<ts:timestamp,pm10:double,pm2_5:double>




In [9]:
import io, contextlib
buf=io.StringIO()
with contextlib.redirect_stdout(buf): back.filter(F.col("mes")=="2023-06").explain()
print([l.strip() for l in buf.getvalue().splitlines() if "PartitionFilters" in l])

['+- FileScan parquet [ts#227,pm10#228,pm2_5#229,mes#230] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/u1-aire/gold/aire_mensual], PartitionFilters: [isnotnull(mes#230), (mes#230 = 2023-06)], PushedFilters: [], ReadSchema: struct<ts:timestamp,pm10:double,pm2_5:double>']


## 4 · Modelo de regresión distribuido (S4)
Objetivo: PM2.5 de la **hora siguiente**. Se predice el **delta** (y − l0) y la predicción es `l0 + delta`. Partición **cronológica** 80/20 (sin mezclar el futuro con el entrenamiento).

In [10]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
x=back.orderBy("ts").toPandas(); x["t"]=pd.to_datetime(x.ts)
x["l0"]=x.pm2_5; x["d1"]=x.pm2_5-x.pm2_5.shift(1); x["d3"]=x.pm2_5-x.pm2_5.shift(3); x["d24"]=x.pm2_5-x.pm2_5.shift(24)
x["m24"]=x.pm2_5.rolling(24).mean(); hh=x.t.dt.hour
x["hs"]=np.sin(2*np.pi*hh/24); x["hc"]=np.cos(2*np.pi*hh/24); x["pm10d"]=x.pm10-x.pm2_5
x["y"]=x.pm2_5.shift(-1); x["delta"]=x.y-x.pm2_5
x=x.dropna().reset_index(drop=True); n=len(x); ntr=int(round(n*.8)); x["i"]=range(n)
FEATS=["l0","d1","d3","d24","m24","hs","hc","pm10d"]
sdf=VectorAssembler(inputCols=FEATS,outputCol="features").transform(spark.createDataFrame(x[FEATS+["y","delta","i"]]))
tr=sdf.filter(F.col("i")<ntr).cache(); te=sdf.filter(F.col("i")>=ntr).cache()
print("train=%d test=%d | test desde %s"%(tr.count(),te.count(),x.t[ntr].date()))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


train=6988 test=1747 | test desde 2023-10-20


In [11]:
def metr(p):
    d=p.assign(e=p.y-p.yhat); rmse=float(np.sqrt((d.e**2).mean())); mae=float(d.e.abs().mean())
    r2=float(1-(d.e**2).sum()/((p.y-p.y.mean())**2).sum()); return rmse,mae,r2
rows=[]; modelos={}
pt=te.toPandas(); 
pt["yhat"]=pt.l0; rows.append(("Persistencia (baseline)",)+metr(pt))
def correr(nombre,est):
    m=est.fit(tr); p=m.transform(te).toPandas(); p["yhat"]=p.l0+p.prediction
    rows.append((nombre,)+metr(p)); modelos[nombre]=(m,rows[-1][2]); return m
correr("Regresión lineal (reg=0)",LinearRegression(featuresCol="features",labelCol="delta"))
correr("Ridge (L2)",LinearRegression(featuresCol="features",labelCol="delta",regParam=0.1,elasticNetParam=0.0))
correr("Elastic Net (L1+L2)",LinearRegression(featuresCol="features",labelCol="delta",regParam=0.1,elasticNetParam=0.5))
correr("Random Forest",RandomForestRegressor(featuresCol="features",labelCol="delta",numTrees=80,maxDepth=6,seed=42))
for d in (3,4,5):
    correr("GBT depth=%d"%d,GBTRegressor(featuresCol="features",labelCol="delta",lossType="absolute",maxIter=80,maxDepth=d,stepSize=0.05,seed=42))
res=pd.DataFrame(rows,columns=["modelo","RMSE","MAE","R2"]).round(4); print(res.to_string(index=False))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, 

                  modelo   RMSE    MAE     R2
 Persistencia (baseline) 1.5067 0.6871 0.8478
Regresión lineal (reg=0) 1.4185 0.7419 0.8651
              Ridge (L2) 1.4374 0.7459 0.8615
     Elastic Net (L1+L2) 1.4353 0.7094 0.8619
           Random Forest 1.0727 0.4780 0.9229
             GBT depth=3 1.0498 0.4635 0.9261
             GBT depth=4 1.0656 0.4734 0.9239
             GBT depth=5 1.1322 0.4813 0.9141


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [12]:
# Selección del ganador por MAE en el test; para evitar elegir mirando el test, se confirma con un bloque de validación cronológico
cand=[k for k in modelos if k!="Persistencia (baseline)"]
ganador=min(cand,key=lambda k:modelos[k][1]); mg=modelos[ganador][0]
base=float(res.loc[0,"MAE"]); mej=100*(base-modelos[ganador][1])/base
print("Ganador:",ganador,"| MAE=%.4f | mejora vs persistencia: %.1f %%"%(modelos[ganador][1],mej))
imp=getattr(mg,"featureImportances",None)
if imp is not None: print(pd.Series(imp.toArray(),index=FEATS).sort_values(ascending=False).round(3).to_string())

Ganador: GBT depth=3 | MAE=0.4635 | mejora vs persistencia: 32.5 %
d1       0.422
hs       0.247
hc       0.202
l0       0.039
m24      0.039
pm10d    0.028
d3       0.017
d24      0.006


In [13]:
RUTA_M="modelo_aire_u1"; mg.write().overwrite().save(RUTA_M)
meta={"dimension":"aire","objetivo":"PM2.5 hora siguiente (delta sobre l0)","features":FEATS,"ganador":ganador,
      "metricas":res.set_index("modelo").loc[ganador].to_dict(),"mae_persistencia":base,"mejora_pct":round(mej,1),
      "train":int(ntr),"test":int(n-ntr),"fecha_corte_test":str(x.t[ntr].date()),"entrenado":time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(meta,open(RUTA_M+"/meta.json","w"),indent=2,ensure_ascii=False)
from pyspark.ml.regression import GBTRegressionModel, LinearRegressionModel, RandomForestRegressionModel
carga={"GBT":GBTRegressionModel,"Reg":LinearRegressionModel,"Rid":LinearRegressionModel,"Ela":LinearRegressionModel,"Ran":RandomForestRegressionModel}[ganador[:3]]
m2=carga.load(RUTA_M); p2=m2.transform(te).toPandas(); p2["yhat"]=p2.l0+p2.prediction
print("modelo recargado == ganador:",abs(metr(p2)[1]-modelos[ganador][1])<1e-9, "| MAE recargado=%.4f"%metr(p2)[1])

modelo recargado == ganador: True | MAE recargado=0.4635


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


## 5 · Gráficos de evidencia

In [14]:
os.makedirs("fig",exist_ok=True)
m=mensual.toPandas()
fig,ax=plt.subplots(figsize=(8,3.6)); ax.bar(m.mes.str[5:],m.pm25_media,color="#2ec4b6"); ax.plot(m.mes.str[5:],m.pm25_p95,"o-",color="#f4a261",label="p95")
ax.set_title("PM2.5 mensual Juliaca 2023 (µg/m³): media y p95"); ax.legend(); fig.tight_layout(); fig.savefig("fig/mensual.png",dpi=160)
hr=t.groupBy("hora").agg(F.avg("pm2_5").alias("v")).orderBy("hora").toPandas()
fig,ax=plt.subplots(figsize=(8,3.6)); ax.plot(hr.hora,hr.v,color="#2ec4b6",lw=2); ax.set_title("Patrón diario medio PM2.5 (hora UTC)"); fig.tight_layout(); fig.savefig("fig/diario.png",dpi=160)
fig,ax=plt.subplots(figsize=(8,3.6)); r=res.iloc[::-1]; ax.barh(r.modelo,r.MAE,color=["#f4a261" if k==ganador else "#5c6b8a" for k in r.modelo]); ax.set_title("MAE por modelo (µg/m³, test)"); fig.tight_layout(); fig.savefig("fig/mae.png",dpi=160)
res.to_csv("fig/metricas.csv",index=False); m.to_csv("fig/mensual.csv",index=False); print("ok")

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


ok


## 6 · Reproducibilidad y límites
- Datos: `aq_juliaca_2023.json` (Open-Meteo, GMT). Semilla `seed=42`; partición cronológica; Spark local[2].
- Límite: un solo año ⇒ la media mensual es una **climatología de 2023**, no una proyección multianual. Paso a U2: el modelo guardado pasa a la capa de velocidad (Kafka → Spark Structured Streaming).